# Netflix Titles Exploratory Data Analysis (EDA)

Proyek analisis data eksploratif untuk mengeksplorasi dataset judul Netflix, dengan tujuan menemukan pola dalam produksi konten, perbandingan film vs serial TV, tren rilis tahunan, distribusi negara produsen, genre terpopuler, serta klasifikasi rating usia dan durasi film.

**Dataset:** `netflix_titles.csv`


### 1. Import Library dan Load Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Atur style visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset menggunakan path relatif dari folder notebook
df = pd.read_csv('../data/netflix_titles.csv')

# Tampilkan 5 data teratas
df.head()

### 2. Cek Dimensi Dataset dan Missing Value Awal

In [ ]:
# Cek jumlah baris dan kolom
print("Ukuran dataset (baris, kolom):", df.shape)

# Cek jumlah data yang duplikat
print("Jumlah duplikat:", df.duplicated().sum())

# Cek missing value di setiap kolom
print("\nJumlah missing values per kolom:")
print(df.isnull().sum())

### 3. Penanganan Missing Values (Data Cleaning)
Kolom teks (`director`, `cast`, `country`) yang bernilai kosong diisi dengan `'Unavailable'`.
Baris yang nilai pentingnya kosong (`rating`, `duration`, `date_added`) dihapus karena jumlahnya sangat sedikit dan penting untuk analisis.

In [ ]:
# Isi nilai kosong di kolom teks dengan 'Unavailable'
df['director'] = df['director'].fillna('Unavailable')
df['cast'] = df['cast'].fillna('Unavailable')
df['country'] = df['country'].fillna('Unavailable')

# Hapus baris yang nilai pentingnya kosong
df = df.dropna(subset=['rating', 'duration', 'date_added']).copy()

# Tampilkan jumlah missing value per kolom setelah penanganan awal
print("Jumlah missing values setelah dibersihkan:")
print(df.isnull().sum())

### 4. Pengecekan dan Penghapusan Duplikat

In [ ]:
# Hapus baris yang duplikat jika ada
df = df.drop_duplicates().copy()

# Cek kembali ukuran dataset
print("Ukuran dataset setelah hapus duplikat:", df.shape)

### 5. Normalisasi dan Parsing Tanggal (`date_added`)
Sebelum diubah ke format datetime, kita hilangkan spasi di awal/akhir string dengan `.str.strip()` terlebih dahulu agar parsing berjalan sempurna tanpa menyisakan nilai kosong (NaT). Setelah itu, kita ekstrak tahun penambahan ke kolom baru `added_year`.

In [ ]:
# Strip spasi lalu ubah kolom 'date_added' ke format datetime
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

# Ekstrak tahun penambahan konten ke Netflix
df['added_year'] = df['date_added'].dt.year

# Cek ulang missing value di date_added
print("Jumlah missing value di date_added setelah perbaikan strip:", df['date_added'].isnull().sum())
df[['title', 'type', 'date_added', 'added_year']].head()

### 6. Ekstrak Durasi: `duration_minutes` (Film) dan `seasons` (TV Show)
Memisahkan kolom durasi sesuai jenis konten agar tidak bercampur:
- `duration_minutes`: Durasi dalam satuan menit khusus untuk Film (Movie).
- `seasons`: Jumlah season khusus untuk Serial TV (TV Show).

In [ ]:
# Ekstrak durasi menit khusus Movie
df['duration_minutes'] = df[df['type'] == 'Movie']['duration'].str.extract(r'(\d+)').astype(float)

# Ekstrak jumlah season khusus TV Show
df['seasons'] = df[df['type'] == 'TV Show']['duration'].str.extract(r'(\d+)').astype(float)

# Tampilkan sampel hasil ekstraksi untuk Movie dan TV Show
print("Contoh data Movie:")
print(df[df['type'] == 'Movie'][['title', 'type', 'duration', 'duration_minutes']].head(3))

print("\nContoh data TV Show:")
print(df[df['type'] == 'TV Show'][['title', 'type', 'duration', 'seasons']].head(3))

### 7. Cek Ulang Dataset Setelah Pembersihan Selesai

In [ ]:
print("Ukuran dataset setelah cleaning:", df.shape)
print("Duplikat setelah cleaning:", df.duplicated().sum())
print("\nJumlah nilai null setelah cleaning:")
print(df.isnull().sum())

---
## Analisis dan Visualisasi Data
Bagian ini menyajikan eksplorasi visual dan analisis temuan utama dari dataset Netflix.

### 4a. Distribusi Film vs TV Show

In [ ]:
# Hitung jumlah dan persentase masing-masing tipe konten
type_counts = df['type'].value_counts()
colors = ['#E50914', '#221F1F']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
sns.barplot(x=type_counts.index, y=type_counts.values, palette=colors, ax=ax1)
ax1.set_title("Jumlah Konten: Movie vs TV Show", fontsize=13, fontweight='bold')
ax1.set_xlabel("Tipe Konten", fontsize=11)
ax1.set_ylabel("Jumlah Judul", fontsize=11)
for i, v in enumerate(type_counts.values):
    ax1.text(i, v + 80, f"{v:,}", ha='center', fontsize=11, fontweight='bold')

# Donut chart
wedges, texts, autotexts = ax2.pie(
    type_counts, labels=type_counts.index, autopct='%1.1f%%',
    colors=colors, startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='white')
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
    at.set_color('white')
ax2.set_title("Proporsi Konten Netflix", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(type_counts)
print("\nPersentase:\n", (type_counts / len(df) * 100).round(1))

> **Insight:**
> Mayoritas konten di platform Netflix adalah Film (Movie) sebanyak 6.126 judul (69,7%), sedangkan Serial TV (TV Show) berjumlah 2.664 judul (30,3%). Hal ini menunjukkan bahwa katalog Netflix didominasi oleh film layar lebar dibandingkan serial televisi.

### 4b. Top 10 Negara Produsen Konten
Kolom `country` memuat beberapa negara yang dipisahkan tanda koma pada konten hasil kolaborasi (co-production). Kita lakukan pemisahan (split) dan ekspansi (explode), serta mengabaikan nilai `'Unavailable'`.

In [ ]:
# Split negara yang dipisah koma dan abaikan 'Unavailable'
countries_series = df[df['country'] != 'Unavailable']['country'].str.split(', ').explode()
top_10_countries = countries_series.value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_countries.values, y=top_10_countries.index, palette='Reds_r')
plt.title("Top 10 Negara Produsen Konten Netflix Terbanyak", fontsize=13, fontweight='bold')
plt.xlabel("Jumlah Judul Konten", fontsize=11)
plt.ylabel("Negara", fontsize=11)

# Tambahkan label angka di samping bar
for i, v in enumerate(top_10_countries.values):
    plt.text(v + 35, i, f"{v:,}", va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Top 10 Negara Produsen:")
print(top_10_countries)

> **Insight:**
> Amerika Serikat merupakan negara produsen konten terbesar di Netflix dengan total 3.680 judul, diikuti oleh India di posisi kedua (1.046 judul) dan Britania Raya di posisi ketiga (803 judul).

### 4c. Tren Jumlah Judul per Tahun (Release Year dan Date Added)
Melihat perkembangan konten berdasarkan tahun rilis resmi (`release_year`) dan tahun konten ditambahkan ke katalog Netflix (`added_year`), dipisahkan antara Film dan Serial TV.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# 1. Tren Tahun Rilis (fokus tahun 2000 ke atas)
recent_releases = df[df['release_year'] >= 2000]
release_trend = recent_releases.groupby(['release_year', 'type']).size().unstack(fill_value=0)

release_trend.plot(kind='line', marker='o', ax=ax1, color=['#E50914', '#221F1F'], linewidth=2)
ax1.set_title("Tren Jumlah Judul Berdasarkan Tahun Rilis (2000–2021)", fontsize=13, fontweight='bold')
ax1.set_xlabel("Tahun Rilis (Release Year)", fontsize=11)
ax1.set_ylabel("Jumlah Judul", fontsize=11)
ax1.legend(title="Tipe")

# 2. Tren Tahun Ditambahkan ke Netflix
added_trend = df.groupby(['added_year', 'type']).size().unstack(fill_value=0)

added_trend.plot(kind='line', marker='s', ax=ax2, color=['#E50914', '#221F1F'], linewidth=2)
ax2.set_title("Tren Penambahan Konten ke Netflix per Tahun", fontsize=13, fontweight='bold')
ax2.set_xlabel("Tahun Ditambahkan (Added Year)", fontsize=11)
ax2.set_ylabel("Jumlah Judul", fontsize=11)
ax2.legend(title="Tipe")

plt.tight_layout()
plt.show()

> **Insight:**
> Produksi konten berdasarkan tahun rilis memuncak pada tahun 2017–2018 untuk film (masing-masing 765 dan 767 judul film). Sementara itu, penambahan konten ke katalog Netflix melonjak drastis sejak tahun 2016 dan mencapai titik puncaknya pada tahun 2019 dengan total lebih dari 2.000 judul baru (1.424 film dan 592 TV show) ditambahkan dalam satu tahun.

### 4d. Top 10 Genre Konten Populer (`listed_in`)
Kolom `listed_in` dapat memuat beberapa kategori sekaligus yang dipisah koma. Kita lakukan split dan explode untuk menghitung frekuensi total setiap genre.

In [ ]:
# Split kolom listed_in dan explode
genres_series = df['listed_in'].str.split(', ').explode()
top_10_genres = genres_series.value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_genres.values, y=top_10_genres.index, palette='Blues_r')
plt.title("Top 10 Genre Konten Terpopuler di Netflix", fontsize=13, fontweight='bold')
plt.xlabel("Jumlah Judul Konten", fontsize=11)
plt.ylabel("Genre", fontsize=11)

# Tambahkan label angka di samping bar
for i, v in enumerate(top_10_genres.values):
    plt.text(v + 25, i, f"{v:,}", va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Top 10 Genre:")
print(top_10_genres)

> **Insight:**
> Tiga genre teratas di Netflix adalah *International Movies* (2.752 judul), *Dramas* (2.426 judul), dan *Comedies* (1.674 judul). Hal ini mencerminkan strategi global Netflix yang memperbanyak konten film internasional serta menyajikan drama dan komedi sebagai daya tarik utama penonton.

### 4e. Distribusi Rating Usia dan Distribusi Durasi Film
Menganalisis klasifikasi usia penonton di seluruh konten serta sebaran panjang durasi khusus film layar lebar.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# 1. Distribusi Rating Usia
rating_counts = df['rating'].value_counts()
sns.barplot(x=rating_counts.values, y=rating_counts.index, palette='viridis', ax=ax1)
ax1.set_title("Distribusi Rating Klasifikasi Usia Konten", fontsize=13, fontweight='bold')
ax1.set_xlabel("Jumlah Konten", fontsize=11)
ax1.set_ylabel("Rating Usia", fontsize=11)

for i, v in enumerate(rating_counts.values):
    ax1.text(v + 25, i, f"{v:,}", va='center', fontsize=9)

# 2. Histogram Durasi Film (Movie)
movie_durations = df[df['type'] == 'Movie']['duration_minutes']
sns.histplot(movie_durations, bins=30, kde=True, color='#E50914', ax=ax2)
ax2.set_title("Distribusi Durasi Film (Menit)", fontsize=13, fontweight='bold')
ax2.set_xlabel("Durasi (Menit)", fontsize=11)
ax2.set_ylabel("Frekuensi Film", fontsize=11)

# Tambahkan garis penanda rata-rata dan median
mean_dur = movie_durations.mean()
median_dur = movie_durations.median()
ax2.axvline(mean_dur, color='blue', linestyle='--', linewidth=1.5, label=f'Rata-rata: {mean_dur:.1f} mnt')
ax2.axvline(median_dur, color='black', linestyle=':', linewidth=1.5, label=f'Median: {median_dur:.0f} mnt')
ax2.legend()

plt.tight_layout()
plt.show()

print("Statistik Deskriptif Durasi Film (menit):")
print(movie_durations.describe().round(1))

> **Insight:**
> Konten Netflix didominasi oleh kategori penonton dewasa dan remaja, ditunjukkan oleh tingginya jumlah rating TV-MA (3.205 judul) dan TV-14 (2.157 judul). Untuk durasi film, distribusinya mendekati normal dengan rata-rata 99,6 menit (~100 menit) dan rentang durasi terbanyak berada di antara 87 hingga 114 menit.

### 8. Ringkasan Kesimpulan
Berdasarkan seluruh hasil analisis data eksploratif di atas:
1. **Proporsi Konten:** Katalog Netflix didominasi oleh Film (69,7% atau 6.126 judul) dibandingkan Serial TV (30,3% atau 2.664 judul).
2. **Produsen Terbesar:** Amerika Serikat memimpin produksi dengan 3.680 judul, diikuti oleh India (1.046 judul) dan Britania Raya (803 judul).
3. **Puncak Pertumbuhan:** Penambahan konten baru ke Netflix melonjak drastis sejak 2016 dan mencapai titik tertinggi pada tahun 2019 (>2.000 judul baru).
4. **Genre Populer:** Kategori paling mendominasi adalah *International Movies* (2.752 judul), *Dramas* (2.426 judul), dan *Comedies* (1.674 judul).
5. **Rating dan Durasi:** Mayoritas konten ditujukan untuk audiens dewasa (TV-MA) dan remaja (TV-14), dengan durasi film rata-rata berkisar di angka 100 menit.